# 01 — jnwb epoch loading and artifacts

Demonstrates **load second** after addressing:

- `jnwb.load_epochs` (SPK / LFP / MUAe)
- `jnwb.save_epoch_artifact`
- `jnwb.load_epoch_artifact`

**Rule:** import-only notebook — no local function or class definitions.

Skips safely when `OMISSION_NWB_ROOT` is unavailable.

In [1]:
import os
import sys
from pathlib import Path

import numpy as np

REPO = Path.cwd()
if not (REPO / "src").exists() and (REPO.parent / "src").exists():
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import jnwb

NWB_ROOT = Path(os.environ.get("OMISSION_NWB_ROOT", "D:/analysis/nwb"))
OUT_DIR = REPO / "outputs" / "jnwb_notebooks"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"NWB root exists: {NWB_ROOT.exists()}")

NWB root exists: True


In [2]:
AFAMILY = ["AAAB", "AXAB", "AAXB", "AAAX"]
WINDOW_MS = (-100, 300)

if not NWB_ROOT.exists():
    print("SKIP: load_epochs requires local NWB data.")
else:
    files = [f for f in jnwb.list_nwb_files(NWB_ROOT) if f.has_spk][:1]
    if not files:
        print("SKIP: no SPK sessions found.")
    else:
        ev = jnwb.address_events(files, conditions=AFAMILY, anchor="p1", correct=True)
        sig = jnwb.address_signals(files, signal="SPK", sessions=ev.sessions, require_area=False, max_items=8)
        epochs = jnwb.load_epochs(files, sig, ev, window_ms=WINDOW_MS, chunk_size=32, bin_ms=1.0)
        batches = list(epochs) if not isinstance(epochs, jnwb.EpochBatch) else [epochs]
        shapes = [np.asarray(b.data).shape for b in batches]
        print(f"load_epochs SPK: {len(batches)} batch(es), shapes={shapes}")

        for signal in ("LFP", "MUAe"):
            analog_files = [f for f in jnwb.list_nwb_files(NWB_ROOT) if (signal == "LFP" and f.has_lfp) or (signal == "MUAe" and f.has_muae)][:1]
            if not analog_files:
                print(f"load_epochs {signal}: SKIP (signal unavailable)")
                continue
            ev_a = jnwb.address_events(analog_files, conditions=["AAXB"], anchor="p1", correct=True)
            sig_a = jnwb.address_signals(analog_files, signal=signal, sessions=ev_a.sessions, max_items=4)
            batch_a = jnwb.load_epochs(analog_files, sig_a, ev_a, window_ms=(-50, 50), chunk_size=1000)
            arr = np.asarray(batch_a.data if isinstance(batch_a, jnwb.EpochBatch) else list(batch_a)[0].data)
            print(f"load_epochs {signal}: shape={arr.shape}  # trial x channel x time")

load_epochs SPK: 2 batch(es), shapes=[(32, 8, 400), (28, 8, 400)]


load_epochs LFP: shape=(6, 4, 100)  # trial x channel x time


load_epochs MUAe: shape=(6, 4, 100)  # trial x channel x time


In [3]:
artifact_npz = OUT_DIR / "demo_spk_epochs.npz"
artifact_manifest = OUT_DIR / "demo_spk_epochs_manifest.json"

if not NWB_ROOT.exists() or "batches" not in globals():
    print("SKIP: save_epoch_artifact (no epochs loaded).")
else:
    meta = jnwb.save_epoch_artifact(
        batches,
        out=artifact_npz,
        manifest=artifact_manifest,
        command="notebook:01_jnwb_epoch_artifacts",
        input_nwb_paths=[f.path for f in files],
    )
    print(f"save_epoch_artifact -> {artifact_npz}")
    print(f"  signal_class={meta.get('signal_class')} schema={meta.get('artifact_schema_version')}")

save_epoch_artifact -> D:\workspace\omission\outputs\jnwb_notebooks\demo_spk_epochs.npz
  signal_class=SPK schema=jnwb_epoch_artifact_v1


In [4]:
if artifact_npz.exists():
    loaded = jnwb.load_epoch_artifact(artifact_npz)
    print(f"load_epoch_artifact shape={np.asarray(loaded.data).shape}")
    print(f"  trials={len(loaded.trial_metadata)} units={len(loaded.signal_metadata)}")
    all_sessions = jnwb.load_epoch_artifact(artifact_npz, load_all_sessions=True)
    if isinstance(all_sessions, list):
        print(f"  load_all_sessions: {len(all_sessions)} session block(s)")
else:
    print("SKIP: load_epoch_artifact (artifact not written).")

load_epoch_artifact shape=(60, 8, 400)


  trials=60 units=8
  load_all_sessions: 1 session block(s)
